In [19]:
!pip -q install -U langchain langchain-google-genai

In [ ]:
import os
os.environ["GOOGLE_API_KEY"]=""

In [21]:
import json

In [22]:
from langchain_core.tools import tool #this is important because we will build or own tools as we go ahead
from langchain_core.messages import HumanMessage, SystemMessage
#SystemMessage will help in defining the behaviour of the LLM and HumanMessage signifies the user query
from langchain_google_genai import ChatGoogleGenerativeAI

In [42]:
#Creating Synthetic Healthcare Data

from typing import List, Dict

# Synthetic symptom categories
symptom_category_keywords = {
    "respiratory": [
        "cough",
        "breathing difficulty",
        "shortness of breath",
        "chest congestion"
    ],
    "cardiac": [
        "chest pain",
        "palpitations",
        "heart rate",
        "dizziness"
    ],
    "infection": [
        "fever",
        "chills",
        "body ache",
        "fatigue"
    ],
    "neurological": [
        "headache",
        "confusion",
        "blurred vision"
    ]
}

# Synthetic severity weights
severity_weights = {
    "breathing difficulty": 30,
    "shortness of breath": 30,
    "chest pain": 35,
    "palpitations": 20,
    "dizziness": 15,
    "fever": 10,
    "high fever": 20,
    "fatigue": 5,
    "confusion": 25,
    "blurred vision": 20,
    "cough": 5
}

In [79]:
#creating a tool for healthcare assessment [Tool - Decorator, Name, Description, Schema]

@tool
def healthcare_assessment_tool(
    symptoms: str,
    age: int,
    additional_info: str = ""
):
    """
    Assess patient symptoms and generate an initial triage evaluation.
    """

    severity_score = 0
    matched_indicators = []

    symptom_text = symptoms.lower()

    for symptom, weight in severity_weights.items():
        if symptom in symptom_text:
            severity_score += weight
            matched_indicators.append(symptom)

    # Age-based adjustment
    if age >= 60:
        severity_score += 10

    # Determine risk level
    if severity_score >= 70:
        risk_level = "HIGH"
    elif severity_score >= 20:
        risk_level = "MEDIUM"
    else:
        risk_level = "LOW"

    human_intervention_required = (
        risk_level == "HIGH"
        or "chest pain" in matched_indicators
        or "breathing difficulty" in matched_indicators
        or "shortness of breath" in matched_indicators
    )

    possible_indicators = list(severity_weights.keys())

    missing_indicators = [
        item for item in possible_indicators
        if item not in matched_indicators
    ][:5]

    if len(matched_indicators) == 0:
      return {
          "severity_score": 0,
          "risk_level": "UNKNOWN",
          "matched_indicators": [],
          "unsupported_symptoms": symptoms,
          "human_intervention_required": False,
          "assessment_status": "NO_SUPPORTED_CATEGORY_MATCH"
      }

    return {
        "severity_score": severity_score,
        "risk_level": risk_level,
        "matched_indicators": matched_indicators,
        "missing_indicators": missing_indicators,
        "human_intervention_required": human_intervention_required
    }

In [75]:
#using gemini llm as the brain for our Agent
llm=ChatGoogleGenerativeAI(
    model="gemini-3-flash-preview"
)

In [34]:
system_prompt = """
You are a Healthcare Patient Triage Assistant.

Responsibilities:
1. Understand patient symptoms and age.
2. Use the healthcare_assessment_tool whenever symptom evaluation is needed.
3. Never provide a medical diagnosis.
4. Keep responses concise and structured.
5. Highlight major risks.
6. Base conclusions only on supplied information.
7. If risk is HIGH or human intervention is required,
   clearly advise immediate medical attention.

Response Format:

Assessment Summary:
Risk Level:
Severity Score:
Key Indicators:
Recommendation:
Disclaimer:
This is not a medical diagnosis and should be reviewed by a healthcare professional.
"""

In [80]:
tools = [healthcare_assessment_tool]

In [81]:
from langgraph.prebuilt import create_react_agent
agent=create_react_agent(llm,tools)

/tmp/ipykernel_2699/3153813159.py:2: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent=create_react_agent(llm,tools)


In [82]:
def run_agent(user_prompt:str)->str:
  result=agent.invoke(
      {
          "messages":[
              SystemMessage(content=system_prompt),
              HumanMessage(content=user_prompt)
          ]
      }
  )
  return result["messages"][-1].content

In [50]:
#test case 1: Low-Risk Scenario
test_case_1=[{
    "prompt":"""
    Patient is 25 years old and has had a mild cough for two days with no breathing difficulty or chest pain.
    """
}]

In [51]:
for i in test_case_1:
  print(run_agent(i["prompt"]))

[{'type': 'text', 'text': 'Assessment Summary:\nThe patient is experiencing a mild cough of short duration (two days) without any critical symptoms such as breathing difficulty or chest pain.\n\nRisk Level: LOW\nSeverity Score: 5/100\nKey Indicators: \n- Presence of cough\n- Absence of breathing difficulty\n- Absence of chest pain\n\nRecommendation:\nMonitor symptoms. Since the risk level is low and no emergency indicators are present, self-care and rest are advised. If symptoms worsen, persist beyond a week, or if breathing difficulties develop, please consult a healthcare provider.\n\nDisclaimer:\nThis is not a medical diagnosis and should be reviewed by a healthcare professional.'}]


In [83]:
#test case 2:  Moderate-Risk Scenario
test_case_2=[{
    "prompt":"""
    Patient is 48 years old and has fever, fatigue and persistent cough for four days.
    """
}]

In [84]:
for i in test_case_2:
  print(run_agent(i["prompt"]))

GoogleRateLimitError: Error calling model 'gemini-3-flash-preview' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3-flash\nPlease retry in 17.498564589s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier

In [61]:
#test case 3:  High-Risk Scenario
test_case_3=[{
    "prompt":"""
    Patient is 67 years old and is experiencing chest pain,
breathing difficulty and dizziness.
    """
}]

In [62]:
for i in test_case_3:
  print(run_agent(i["prompt"]))

[{'type': 'text', 'text': 'Assessment Summary:\nThe patient is experiencing high-risk symptoms including chest pain and respiratory distress, which require urgent evaluation.\n\nRisk Level: HIGH\nSeverity Score: 90\nKey Indicators: Chest pain, breathing difficulty, dizziness\nRecommendation: Seek immediate emergency medical attention or call emergency services (e.g., 911) right away.\n\nDisclaimer:\nThis is not a medical diagnosis and should be reviewed by a healthcare professional.', 'extras': {'signature': 'EpsFCpgFARFNMg+1XKI4sTfYQljWzEUrYMP7BQjo7Q6Ilhjz+HaAASIclcVW7nrvn14Du+/WUKeClLmVYpNlW8qP4jviiHvzFBFOl+a5YhXXfId2EPXZSTk2xnz9a7Eg41Izw1/3YEAI9Q8fRaPtS+dgRrHImREGZr1wAKGS8PVDu6DIh55eddl3yYtYMU9c5jvKTFvvnyuYnx9nYQehnBumaSn0A32UNB4mf/cJ3Myhh5LqisxZD9/4/ZLbiCIkd3rij+2UJfpmufGpyu8t0jGQKw+Wln1LPuxyBsOpNdfvBs7aFZB1MHLU299n0/SohJf31itDjVv9p10OZQzd1fZwlFaFMZmr2L55rYvksflzfQv9ZLm2Pnpb8jpcmikg4QSGDD7v61oMc4AWs4yAXqlEu4tM6JWCc6CfDWNvN/dgCm56FHY/YY3lakta+w4sopxzg6Rw2zEtSn1EkT8ts8iOD2jBE4ap2SGoK

In [72]:
#Scenario A - Irrelevant Symptom Category
scenario_1={
    "prompt":"""
    Patient is 35 years old and reports dry skin, hair loss,
and occasional nail brittleness for several months.
    """
}

In [78]:
print(run_agent(scenario_1["prompt"]))

GoogleRateLimitError: Error calling model 'gemini-3-flash-preview' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3-flash\nPlease retry in 31.035213532s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier

In [67]:
#Scenario B — Incomplete Patient Information
scenario_2={
    "prompt":"""
    Patient has chest pain and dizziness.
    """
}

In [68]:
print(run_agent(scenario_2["prompt"]))

[{'type': 'text', 'text': "Please provide the patient's age so that I can provide an accurate assessment.", 'extras': {'signature': 'ErcGCrQGARFNMg9BzrFbSO1vb8e4ur0qnMPWkdlHfHYPOC2VROt+1aBy7p66ePVBjHprfjJ2DaHgE5jhlGa5AEn9wsNHz+MQOcEKZh7y23oBYgwbEdUhbFj+3xrySXAi1os7ZHiNoq4CsVWYWbRgu4BlX0aVPHCSbCP62cS9VcNne0F+1cl+Z0WBgwy4T5X1wFw5Uyn+0C+C6wfocTarWVktkbdwpg+1d87z5lMAAsqN0OSYHjV6BHGPvDIXMFyfZ9cVzHDcY7lv+qOSswO/SE+ip0rLb00rne0FHJiG2Euztk6Gt/IPtaP31D3nafWxT9TA/ShfVT29S14/6WoF2P5TLh5dneXGz/4cQ50t7tfrULt5N2jw7VW9Agw27Gcyq6iHSgytfbcJuE/trCipzgA72bOkrkihiMqaWY6o2g+BG4IvSJWEC2toVSBbkQY/FuFLIB5axO5RXTuoKKb7hUA4Nqq5DXrRfnPBqV/he7czLeWY/eKLIxevHMx/FxnVXG4gYoDvILm9EY+SO2DoqcHI8mTWFVxxHzJFXzZYBhyee4fcfcSxlnTuEBSQYLc+hRnz/ESXgFjFkHr1qZ9pW/FJheGkoQtdv4SMoEv0cC51xLoLaT8h8KY6VyptH1svRg9q2dD41osDncxPgsWZgmyztiZlHaMe6LcvD0fjkOc6Y6jy3RW9zAg8CDhrrrywWsGdy3lIt4gleRTeqq9vk49/nNet9VPxzYMUJZMSo/p8CU5gHPNXEkVEtdWhrNqagKdPZ9RRZgxtHjYOr9J9ISjiPbZUT8JDlbMx0FHhb73sllYQ011L394VnDXSJ1A1LaX7zGdgdg9yLdczp7pHY8mFTNrtlPut5sn

In [69]:
#Scenario C — Incomplete Patient Information
scenario_3={
    "prompt":"""
    Patient is 58 years old and reports chest pain,
fever, shortness of breath, fatigue, and dizziness
for the last two days.
    """
}

In [70]:
print(run_agent(scenario_3["prompt"]))

[{'type': 'text', 'text': 'Assessment Summary:\nThe patient is experiencing a combination of high-risk symptoms including chest pain, shortness of breath, and dizziness, alongside fever and fatigue. Given the severity score and the nature of the symptoms, immediate evaluation is necessary.\n\nRisk Level: HIGH\nSeverity Score: 95/100\n\nKey Indicators:\n- Chest pain\n- Shortness of breath\n- Dizziness\n- Fever\n- Fatigue\n\nRecommendation:\nSeek immediate emergency medical attention or call emergency services (e.g., 911) right away. Human intervention is required due to the high risk level and severity of symptoms.\n\nDisclaimer:\nThis is not a medical diagnosis and should be reviewed by a healthcare professional.'}]
